# Retail Sales Performance

Review sales and profit by region, category, product and channel, then track monthly performance.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
df = pd.read_csv(data_file, parse_dates=['Date'])
display(df.head())
print('Shape:', df.shape)
# Resolve the data folder whether Jupyter starts in the repository root or notebooks/.
data_file = Path('data/retail_sales_50000.csv')
if not data_file.exists():
    data_file = Path('../data/retail_sales_50000.csv')


In [ ]:
print('Missing values by column:\n', df.isna().sum())
print('Duplicate rows:', df.duplicated().sum())
print('Date range:', df['Date'].min().date(), 'to', df['Date'].max().date())
df = df.drop_duplicates().copy()
kpis = {
    'Transactions': len(df),
    'Units Sold': df['Units_Sold'].sum(),
    'Revenue': df['Revenue'].sum(),
    'Profit': df['Profit'].sum(),
    'Profit Margin %': df['Profit'].sum() / df['Revenue'].sum() * 100
}
display(pd.Series(kpis))

In [ ]:
region = df.groupby('Region').agg(Units_Sold=('Units_Sold', 'sum'), Revenue=('Revenue', 'sum'), Profit=('Profit', 'sum')).sort_values('Revenue', ascending=False)
category = df.groupby('Category').agg(Units_Sold=('Units_Sold', 'sum'), Revenue=('Revenue', 'sum'), Profit=('Profit', 'sum')).sort_values('Revenue', ascending=False)
product = df.groupby('Product').agg(Units_Sold=('Units_Sold', 'sum'), Revenue=('Revenue', 'sum'), Profit=('Profit', 'sum')).sort_values('Revenue', ascending=False)
channel = df.groupby('Channel').agg(Transactions=('Date', 'size'), Revenue=('Revenue', 'sum'), Profit=('Profit', 'sum')).sort_values('Revenue', ascending=False)
for table in (region, category, product.head(10), channel):
    table['Profit_Margin_Pct'] = table['Profit'] / table['Revenue'] * 100
display(region)
display(category)
display(product.head(10))
display(channel)

In [ ]:
monthly = df.groupby(df['Date'].dt.to_period('M')).agg(Revenue=('Revenue', 'sum'), Profit=('Profit', 'sum')).reset_index()
monthly['Date'] = monthly['Date'].dt.to_timestamp()
monthly['Revenue_MoM_Pct'] = monthly['Revenue'].pct_change() * 100
monthly['Profit_MoM_Pct'] = monthly['Profit'].pct_change() * 100
display(monthly.tail(12))
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(monthly['Date'], monthly['Revenue'])
ax.set(title='Monthly revenue', xlabel='Month', ylabel='Revenue')
fig.tight_layout()
plt.show()

## Interpretation

Compare profit margin as well as revenue. A high-revenue product or channel is not necessarily the most profitable. Month-over-month percentage changes can be unstable when the previous month has zero revenue or profit; inspect the underlying totals as well.

## Power BI handoff

Suggested KPIs: revenue, profit, profit margin and units sold. Add monthly trends, product and category rankings, regional performance and channel comparisons.